<a href="https://colab.research.google.com/github/HitanshuGedam/quantum-learning-journey/blob/main/Day8_POVMs_Naimarks_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 8: POVMs - Positive Operator-Valued Measures

## From Projective Measurements to Generalized Measurements

## Learning Objectives

By the end of this notebook, you will understand:

1. The limitations of projective measurements
2. What POVMs are and why they are needed
3. The mathematical definition of POVM elements
4. Naimark's theorem (dilation theorem)
5. How to implement POVMs in QuTiP
6. Applications of POVMs in quantum information

## Philosophical Motivation

Projective measurements (von Neumann measurements) are ideal but limited. In real experiments, we often face:
- Imperfect detectors
- Limited measurement capabilities
- The need to measure non-orthogonal states

POVMs (Positive Operator-Valued Measures) provide a more general framework for describing quantum measurements, including those that are not projective.


## References

- Nielsen & Chuang (2010). Quantum Computation and Quantum Information. Chapter 2.2.
- QuTiP Documentation: https://qutip.org/
- https://en.wikipedia.org/wiki/POVM

In [1]:
# ============================================================================
# SETUP AND INSTALLATIONS
# ============================================================================

!pip install qutip qutip_qip -q

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

print(f"NumPy version: {np.__version__}")
print(f"QuTiP version: {qt.__version__}")
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.8/140.8 kB 7.1 MB/s eta 0:00:00
NumPy version: 2.0.2
QuTiP version: 5.2.3
Setup complete.


## 1. Limitations of Projective Measurements

### Projective Measurements Recap

Projective measurements are defined by a set of orthogonal projectors $\{P_m\}$ satisfying:

$$ P_m P_n = \delta_{mn} P_m, \quad \sum_m P_m = I, \quad P_m^2 = P_m $$

### Limitations

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Limitation</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Orthogonality</td>
            <td style="padding: 8px;">Outcomes must correspond to orthogonal states</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Number of outcomes</td>
            <td style="padding: 8px;">At most $d$ outcomes (where $d$ is Hilbert space dimension)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Repeatability</td>
            <td style="padding: 8px;">Measuring twice gives the same result</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Idealized</td>
            <td style="padding: 8px;">Assumes perfect, noise-free detectors</td>
        </tr>
    </tbody>
</table>

### When Projective Measurements Are Insufficient

- **Distinguishing non-orthogonal states:** Projective measurements cannot perfectly distinguish non-orthogonal states
- **Unsharp measurements:** Measurements with limited precision (e.g., position measurement)
- **Weak measurements:** Measurements that only slightly disturb the state
- **Real detectors:** Detectors have finite efficiency and noise

### Example: Distinguishing Non-Orthogonal States

Consider two non-orthogonal states:

$$ |\psi_1\rangle = |0\rangle, \quad |\psi_2\rangle = \cos\theta|0\rangle + \sin\theta|1\rangle $$

No projective measurement can perfectly distinguish these states because $\langle\psi_1|\psi_2\rangle \neq 0$.

POVMs can achieve optimal discrimination with some probability of error.

In [4]:
# ============================================================================
# NON-ORTHOGONAL STATES - PROJECTIVE MEASUREMENT LIMITATION
# ============================================================================

print("=" * 70)
print("NON-ORTHOGONAL STATES")
print("=" * 70)

# Define two non-orthogonal states
theta = np.pi/6  # 30 degrees
cos_t = np.cos(theta)
sin_t = np.sin(theta)

psi1 = qt.basis(2, 0)
psi2 = cos_t * qt.basis(2, 0) + sin_t * qt.basis(2, 1)
psi2 = psi2.unit()

print(f"\nState |ψ₁⟩ = {psi1}")
print(f"State |ψ₂⟩ = {psi2}")

# Calculate overlap correctly - psi1.dag() * psi2 returns a complex number
inner_product = (psi1.dag() * psi2)
# For a 1x1 matrix, we can access the element directly
if hasattr(inner_product, 'full'):
    overlap = abs(inner_product.full()[0, 0])**2
else:
    overlap = abs(inner_product)**2
print(f"Overlap |⟨ψ₁|ψ₂⟩|² = {overlap:.4f}")

# Projective measurement in computational basis
P0 = qt.basis(2, 0) * qt.basis(2, 0).dag()
P1 = qt.basis(2, 1) * qt.basis(2, 1).dag()

prob_psi1_0 = (psi1.dag() * P0 * psi1).real
prob_psi1_1 = (psi1.dag() * P1 * psi1).real

prob_psi2_0 = (psi2.dag() * P0 * psi2).real
prob_psi2_1 = (psi2.dag() * P1 * psi2).real

print(f"\nProjective measurement probabilities:")
print(f"  For |ψ₁⟩: P(0) = {prob_psi1_0:.3f}, P(1) = {prob_psi1_1:.3f}")
print(f"  For |ψ₂⟩: P(0) = {prob_psi2_0:.3f}, P(1) = {prob_psi2_1:.3f}")

print("\n⚠️ Cannot perfectly distinguish these states with projective measurement!")
print("   Both have non-zero probability for outcome 0.")

NON-ORTHOGONAL STATES

State |ψ₁⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]]
State |ψ₂⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.8660254]
 [0.5      ]]
Overlap |⟨ψ₁|ψ₂⟩|² = 0.7500

Projective measurement probabilities:
  For |ψ₁⟩: P(0) = 1.000, P(1) = 0.000
  For |ψ₂⟩: P(0) = 0.750, P(1) = 0.250

⚠️ Cannot perfectly distinguish these states with projective measurement!
   Both have non-zero probability for outcome 0.


## 2. POVM Definition

### What is a POVM?

A POVM (Positive Operator-Valued Measure) is a set of operators $\{E_m\}$ that satisfy:

$$ E_m \geq 0 \quad \text{(Positive semidefinite)} $$

$$ \sum_m E_m = I \quad \text{(Completeness)} $$

### Key Differences from Projective Measurements

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Feature</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Projective</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">POVM</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Operators</td>
            <td style="padding: 8px;">Projectors $P_m$</td>
            <td style="padding: 8px;">Positive operators $E_m$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Condition</td>
            <td style="padding: 8px;">$P_m P_n = \delta_{mn} P_m$</td>
            <td style="padding: 8px;">No orthogonality required</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Completeness</td>
            <td style="padding: 8px;">$\sum_m P_m = I$</td>
            <td style="padding: 8px;">$\sum_m E_m = I$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Number of outcomes</td>
            <td style="padding: 8px;">≤ dimension</td>
            <td style="padding: 8px;">Can exceed dimension</td>
        </tr>
        <tr>
            <td style="padding: 8px;">State after measurement</td>
            <td style="padding: 8px;">Defined collapse</td>
            <td style="padding: 8px;">Not specified</td>
        </tr>
    </tbody>
</table>

### Probability Rule

For a quantum state $\rho$, the probability of outcome $m$ is:

$$ p(m) = \text{Tr}(E_m \rho) $$

### Example: 3-Outcome POVM for Qubit

For a qubit, we can have a POVM with 3 outcomes:

$$ E_0 = \frac{2}{3}|0\rangle\langle0|, \quad E_1 = \frac{2}{3}|1\rangle\langle1|, \quad E_2 = \frac{2}{3}|+\rangle\langle+| $$

These satisfy $\sum_m E_m = I$ but are not projectors.

In [5]:
# ============================================================================
# SIMPLE POVM EXAMPLE
# ============================================================================

print("=" * 70)
print("SIMPLE POVM EXAMPLE")
print("=" * 70)

# Define a simple 3-outcome POVM for qubit
# E0 = a|0⟩⟨0|, E1 = a|1⟩⟨1|, E2 = I - E0 - E1

# Choose a such that all E_m are positive
a = 0.5

E0 = a * (qt.basis(2, 0) * qt.basis(2, 0).dag())
E1 = a * (qt.basis(2, 1) * qt.basis(2, 1).dag())
E2 = qt.qeye(2) - E0 - E1

print(f"\nPOVM elements:")
print(f"E0 = {E0}")
print(f"E1 = {E1}")
print(f"E2 = {E2}")

# Verify completeness
sum_E = E0 + E1 + E2
print(f"\nSum E_m = {sum_E}")
print(f"Sum = I? {np.allclose(sum_E.full(), qt.qeye(2).full())}")

# Check positivity (eigenvalues)
eig0 = E0.eigenenergies()
eig1 = E1.eigenenergies()
eig2 = E2.eigenenergies()

print(f"\nEigenvalues:")
print(f"  E0: {eig0}")
print(f"  E1: {eig1}")
print(f"  E2: {eig2}")

# Test on different states
psi0 = qt.basis(2, 0)
psi1 = qt.basis(2, 1)
psi_plus = (qt.basis(2, 0) + qt.basis(2, 1)).unit()

print(f"\nProbabilities for |0⟩:")
p0_0 = (psi0.dag() * E0 * psi0).real
p0_1 = (psi0.dag() * E1 * psi0).real
p0_2 = (psi0.dag() * E2 * psi0).real
print(f"  p0 = {p0_0:.3f}")
print(f"  p1 = {p0_1:.3f}")
print(f"  p2 = {p0_2:.3f}")
print(f"  Sum = {p0_0 + p0_1 + p0_2:.3f}")

print(f"\nProbabilities for |1⟩:")
p1_0 = (psi1.dag() * E0 * psi1).real
p1_1 = (psi1.dag() * E1 * psi1).real
p1_2 = (psi1.dag() * E2 * psi1).real
print(f"  p0 = {p1_0:.3f}")
print(f"  p1 = {p1_1:.3f}")
print(f"  p2 = {p1_2:.3f}")
print(f"  Sum = {p1_0 + p1_1 + p1_2:.3f}")

print(f"\nProbabilities for |+⟩:")
p_plus_0 = (psi_plus.dag() * E0 * psi_plus).real
p_plus_1 = (psi_plus.dag() * E1 * psi_plus).real
p_plus_2 = (psi_plus.dag() * E2 * psi_plus).real
print(f"  p0 = {p_plus_0:.3f}")
print(f"  p1 = {p_plus_1:.3f}")
print(f"  p2 = {p_plus_2:.3f}")
print(f"  Sum = {p_plus_0 + p_plus_1 + p_plus_2:.3f}")

print("\n✅ POVM elements sum to identity and are positive!")

SIMPLE POVM EXAMPLE

POVM elements:
E0 = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.5 0. ]
 [0.  0. ]]
E1 = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.  0. ]
 [0.  0.5]]
E2 = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.5 0. ]
 [0.  0.5]]

Sum E_m = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 1.]]
Sum = I? True

Eigenvalues:
  E0: [0.  0.5]
  E1: [0.  0.5]
  E2: [0.5 0.5]

Probabilities for |0⟩:
  p0 = 0.500
  p1 = 0.000
  p2 = 0.500
  Sum = 1.000

Probabilities for |1⟩:
  p0 = 0.000
  p1 = 0.500
  p2 = 0.500
  Sum = 1.000

Probabilities for |+⟩:
  p0 = 0.250
  p1 = 0.250
  p2 = 0.500
  Sum = 1.000

✅ POVM elements sum to identity and are positive!


## 3. Naimark's Theorem (Dilation Theorem)

### The Statement

Naimark's theorem states that any POVM can be realized as a projective measurement on a larger Hilbert space (by adding an ancilla).

### Mathematical Formulation

Given a POVM $\{E_m\}$ on a Hilbert space $\mathcal{H}$, there exists:

1. An ancilla Hilbert space $\mathcal{H}_A$
2. A pure state $|0\rangle_A$ on $\mathcal{H}_A$
3. A unitary operator $U$ on $\mathcal{H} \otimes \mathcal{H}_A$
4. Projectors $P_m$ on $\mathcal{H} \otimes \mathcal{H}_A$

such that:

$$ E_m = \langle 0|_A U^\dagger P_m U |0\rangle_A $$

### Physical Interpretation

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Concept</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Ancilla</td>
            <td style="padding: 8px;">An extra quantum system (ancillary qubit)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Unitary</td>
            <td style="padding: 8px;">Entangles system with ancilla</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Projective measurement</td>
            <td style="padding: 8px;">Measurement on the combined system</td>
        </tr>
        <tr>
            <td style="padding: 8px;">POVM</td>
            <td style="padding: 8px;">Effective measurement on original system</td>
        </tr>
    </tbody>
</table>

### Why This Matters

- **Practical implementation:** Any POVM can be implemented using projective measurements + ancillas
- **Theoretical insight:** Projective measurements are sufficient if we allow ancillas
- **Resource cost:** Tells us how many ancilla qubits are needed

### Example

A 3-outcome POVM on a qubit requires:
- Ancilla dimension = 3 (or 2 qubits)
- Total Hilbert space dimension = 2 × 3 = 6
- Projective measurement on 6D space gives POVM on 2D space

In [6]:
# ============================================================================
# NAIMARK'S THEOREM DEMONSTRATION
# ============================================================================

print("=" * 70)
print("NAIMARK'S THEOREM DEMONSTRATION")
print("=" * 70)

print("""
Naimark's Theorem: Any POVM can be realized as a projective measurement
on a larger Hilbert space (system + ancilla).

Example: A 3-outcome POVM on a qubit can be realized by:
1. Add one ancilla qubit (total Hilbert space dimension 4)
2. Apply a unitary U that entangles system and ancilla
3. Perform projective measurement on the combined system
4. Trace out ancilla to get POVM probabilities
""")

# Simple example: 2-outcome POVM (already projective, no ancilla needed)
print("\n" + "=" * 50)
print("PROJECTIVE MEASUREMENT (No ancilla needed)")
print("=" * 50)

# Define a projective measurement (already a POVM)
P0 = qt.basis(2, 0) * qt.basis(2, 0).dag()
P1 = qt.basis(2, 1) * qt.basis(2, 1).dag()

print(f"P0 = {P0}")
print(f"P1 = {P1}")
print(f"P0 + P1 = I? {np.allclose((P0 + P1).full(), qt.qeye(2).full())}")

# For a 3-outcome POVM, we need an ancilla
print("\n" + "=" * 50)
print("3-OUTCOME POVM (Requires ancilla)")
print("=" * 50)

print("""
A 3-outcome POVM on a qubit:
  - Requires ancilla dimension = 3 (or 2 qubits)
  - Total Hilbert space dimension = 2 × 3 = 6
  - Projective measurement on 6D space gives POVM on 2D space

Naimark's theorem guarantees that such a dilation exists,
but constructing the explicit unitary can be complex.
""")

print("\n✅ Naimark's theorem guarantees existence of such dilation!")
print("   Any POVM can be implemented as a projective measurement + ancilla.")

NAIMARK'S THEOREM DEMONSTRATION

Naimark's Theorem: Any POVM can be realized as a projective measurement
on a larger Hilbert space (system + ancilla).

Example: A 3-outcome POVM on a qubit can be realized by:
1. Add one ancilla qubit (total Hilbert space dimension 4)
2. Apply a unitary U that entangles system and ancilla
3. Perform projective measurement on the combined system
4. Trace out ancilla to get POVM probabilities


PROJECTIVE MEASUREMENT (No ancilla needed)
P0 = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 0.]]
P1 = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0.]
 [0. 1.]]
P0 + P1 = I? True

3-OUTCOME POVM (Requires ancilla)

A 3-outcome POVM on a qubit:
  - Requires ancilla dimension = 3 (or 2 qubits)
  - Total Hilbert space dimension = 2 × 3 = 6
  - Projective measurement on 6D space gives POVM on 2D space

Naimark's theorem guarantees that such a dilation

## 4. Implementing POVMs in QuTiP

### QuTiP POVM Implementation

QuTiP provides tools for implementing POVMs via the `qutip.measurement` module.

### Steps to Implement a POVM

1. Define the POVM elements $\{E_m\}$
2. Verify completeness: $\sum_m E_m = I$
3. Verify positivity: $E_m \geq 0$ (all eigenvalues non-negative)
4. Calculate probabilities: $p(m) = \text{Tr}(E_m \rho)$

### Example: 3-Outcome POVM

A simple 3-outcome POVM for qubit:

$$ E_0 = a|0\rangle\langle0|, \quad E_1 = a|1\rangle\langle1|, \quad E_2 = I - E_0 - E_1 $$

where $0 \leq a \leq 1$ ensures positivity.

In [16]:
# ============================================================================
# IMPLEMENTING POVMS IN QuTiP USING qutip.measurement
# ============================================================================

from qutip.measurement import measure_povm, measurement_statistics_povm

print("=" * 70)
print("IMPLEMENTING POVMS IN QuTiP")
print("=" * 70)

# Define basis states
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)

# ============================================================================
# SIMPLE 2-OUTCOME POVM (Non-projective)
# ============================================================================
print("\n" + "=" * 70)
print("SIMPLE 2-OUTCOME NON-PROJECTIVE POVM")
print("=" * 70)

# Define a non-projective POVM
# E0 = 0.6|0⟩⟨0| + 0.4|1⟩⟨1|
# E1 = I - E0

E0 = 0.6 * (ket0 * ket0.dag()) + 0.4 * (ket1 * ket1.dag())
E1 = qt.qeye(2) - E0

print("\nPOVM elements:")
print(f"  E0 =")
print(E0)
print(f"\n  E1 =")
print(E1)

# Verify completeness
sum_E = E0 + E1
print(f"\nE0 + E1 =")
print(sum_E)
print(f"E0 + E1 = I? {np.allclose(sum_E.full(), qt.qeye(2).full())}")

# Check positivity (eigenvalues should be >= 0)
eig0 = E0.eigenenergies()
eig1 = E1.eigenenergies()
print(f"\nEigenvalues of E0: {eig0}")
print(f"Eigenvalues of E1: {eig1}")

if np.all(eig0 >= -1e-10) and np.all(eig1 >= -1e-10):
    print("✅ Both POVM elements are positive semidefinite")

povm_ops = [E0, E1]

# Test on different states
print(f"\n" + "=" * 50)
print("Testing POVM on |0⟩")
print("=" * 50)

# Use measurement_statistics_povm
try:
    outcomes, probs = measurement_statistics_povm(ket0, povm_ops)
    print(f"\nNumber of outcomes: {len(outcomes)}")
    for idx, (outcome, prob) in enumerate(zip(outcomes, probs)):
        prob_val = prob.full().real[0, 0] if hasattr(prob, 'full') else float(prob)
        print(f"  Outcome {idx}: probability = {prob_val:.3f}")
except Exception as e:
    print(f"Error: {e}")
    print("\nTrying alternative method...")
    # Manual probability calculation
    for idx, E in enumerate(povm_ops):
        prob = (ket0.dag() * E * ket0).real
        print(f"  Outcome {idx}: probability = {prob:.3f}")

print(f"\n" + "=" * 50)
print("Testing POVM on |1⟩")
print("=" * 50)

try:
    outcomes, probs = measurement_statistics_povm(ket1, povm_ops)
    for idx, (outcome, prob) in enumerate(zip(outcomes, probs)):
        prob_val = prob.full().real[0, 0] if hasattr(prob, 'full') else float(prob)
        print(f"  Outcome {idx}: probability = {prob_val:.3f}")
except Exception as e:
    for idx, E in enumerate(povm_ops):
        prob = (ket1.dag() * E * ket1).real
        print(f"  Outcome {idx}: probability = {prob:.3f}")

print(f"\n" + "=" * 50)
print("Testing POVM on |+⟩")
print("=" * 50)

ket_plus = (ket0 + ket1).unit()
try:
    outcomes, probs = measurement_statistics_povm(ket_plus, povm_ops)
    for idx, (outcome, prob) in enumerate(zip(outcomes, probs)):
        prob_val = prob.full().real[0, 0] if hasattr(prob, 'full') else float(prob)
        print(f"  Outcome {idx}: probability = {prob_val:.3f}")
except Exception as e:
    for idx, E in enumerate(povm_ops):
        prob = (ket_plus.dag() * E * ket_plus).real
        print(f"  Outcome {idx}: probability = {prob:.3f}")

print("\n✅ POVM implementation verified manually!")
print("   - POVM elements are positive (eigenvalues >= 0)")
print("   - E0 + E1 = I")
print("   - Probabilities sum to 1")

IMPLEMENTING POVMS IN QuTiP

SIMPLE 2-OUTCOME NON-PROJECTIVE POVM

POVM elements:
  E0 =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.6 0. ]
 [0.  0.4]]

  E1 =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.4 0. ]
 [0.  0.6]]

E0 + E1 =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 1.]]
E0 + E1 = I? True

Eigenvalues of E0: [0.4 0.6]
Eigenvalues of E1: [0.4 0.6]
✅ Both POVM elements are positive semidefinite

Testing POVM on |0⟩
Error: measurement operators must sum to identity

Trying alternative method...
  Outcome 0: probability = 0.600
  Outcome 1: probability = 0.400

Testing POVM on |1⟩
  Outcome 0: probability = 0.400
  Outcome 1: probability = 0.600

Testing POVM on |+⟩
  Outcome 0: probability = 0.500
  Outcome 1: probability = 0.500

✅ POVM implementation verified manually!
   - POVM elements are positive (ei

## 5. Applications of POVMs

### Key Applications

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Application</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Unambiguous State Discrimination</td>
            <td style="padding: 8px;">Distinguish non-orthogonal states without error</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Quantum Cryptography</td>
            <td style="padding: 8px;">QKD protocols like BB84 use POVMs for eavesdropping detection</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Weak Measurements</td>
            <td style="padding: 8px;">Measurements with minimal disturbance to the state</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Quantum Tomography</td>
            <td style="padding: 8px;">Reconstruct quantum states from measurement data</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Entanglement Detection</td>
            <td style="padding: 8px;">Detect entanglement using POVMs</td>
        </tr>
    </tbody>
</table>

### Unambiguous State Discrimination (USD)

Given two non-orthogonal states $|\psi_1\rangle$ and $|\psi_2\rangle$, a USD POVM has:

- Outcome 1: Definitely $|\psi_1\rangle$
- Outcome 2: Definitely $|\psi_2\rangle$
- Outcome ?: Inconclusive

**Key property:** No errors, but sometimes no answer!

### Connection to Quantum Cryptography

In BB84 protocol:
- Alice sends qubits in random bases
- Bob measures in random bases
- When bases match, they get a key bit
- POVMs can detect eavesdropping by measuring in optimal bases

### Weak Measurements

Weak measurements extract partial information with minimal disturbance.

Useful for:
- Quantum state tracking
- Feedback control
- Quantum thermodynamics

### Key Insight

POVMs provide the most general description of quantum measurements and are essential for:
- Optimal state discrimination
- Quantum information processing with real detectors
- Understanding fundamental limits of quantum measurements

### Reference
- https://www.youtube.com/watch?v=tc5YURMuzzw

In [18]:
# ============================================================================
# UNANIMOUS STATE DISCRIMINATION WITH POVM
# ============================================================================

print("=" * 70)
print("UNANIMOUS STATE DISCRIMINATION")
print("=" * 70)

# Define states to discriminate
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)

theta = np.pi/6  # 30 degrees
cos_t = np.cos(theta)
sin_t = np.sin(theta)

psi_a = ket0
psi_b = cos_t * ket0 + sin_t * ket1
psi_b = psi_b.unit()

print(f"\nStates to discriminate:")
print(f"  |ψ₁⟩ = {psi_a}")
print(f"  |ψ₂⟩ = {psi_b}")

# Calculate overlap correctly - the result is a complex number
inner_product = psi_a.dag() * psi_b
# inner_product is already a complex number
if hasattr(inner_product, 'full'):
    overlap = abs(inner_product.full()[0, 0])**2
else:
    overlap = abs(inner_product)**2
print(f"  Overlap |⟨ψ₁|ψ₂⟩|² = {overlap:.4f}")

# Find vectors orthogonal to each state
# |ψ₁_perp⟩ is orthogonal to |ψ₁⟩
psi_a_perp = ket1  # |1⟩ is orthogonal to |0⟩

# |ψ₂_perp⟩ is orthogonal to |ψ₂⟩
# For |ψ₂⟩ = cosθ|0⟩ + sinθ|1⟩, the orthogonal state is sinθ|0⟩ - cosθ|1⟩
psi_b_perp = sin_t * ket0 - cos_t * ket1
psi_b_perp = psi_b_perp.unit()

print(f"\nOrthogonal states:")
print(f"  |ψ₁_perp⟩ = {psi_a_perp}")
print(f"  |ψ₂_perp⟩ = {psi_b_perp}")

# Projectors onto orthogonal subspaces
Pa = psi_a_perp * psi_a_perp.dag()
Pb = psi_b_perp * psi_b_perp.dag()

# Optimal scaling factor for USD
# alpha = 1/(1 + |⟨ψ₁|ψ₂⟩|)
overlap_abs = np.sqrt(overlap)
alpha = 1 / (1 + overlap_abs)
print(f"\nOptimal scaling factor α = {alpha:.4f}")

# POVM elements
Ea = alpha * Pa
Eb = alpha * Pb
Ec = qt.qeye(2) - Ea - Eb

print(f"\nUSD POVM elements:")
print(f"  Ea (identifies |ψ₁⟩) =")
print(Ea)
print(f"\n  Eb (identifies |ψ₂⟩) =")
print(Eb)
print(f"\n  Ec (inconclusive) =")
print(Ec)

# Verify completeness
sum_E = Ea + Eb + Ec
print(f"\nEa + Eb + Ec = I? {np.allclose(sum_E.full(), qt.qeye(2).full())}")

# Verify positivity
eig_a = Ea.eigenenergies()
eig_b = Eb.eigenenergies()
eig_c = Ec.eigenenergies()
print(f"\nEigenvalues:")
print(f"  Ea: {eig_a}")
print(f"  Eb: {eig_b}")
print(f"  Ec: {eig_c}")

if np.all(eig_a >= -1e-10) and np.all(eig_b >= -1e-10) and np.all(eig_c >= -1e-10):
    print("✅ All POVM elements are positive semidefinite")

# Test on |ψ₁⟩
print(f"\n" + "=" * 50)
print("Testing on |ψ₁⟩")
print("=" * 50)

prob_a1 = (psi_a.dag() * Ea * psi_a).real
prob_b1 = (psi_a.dag() * Eb * psi_a).real
prob_c1 = (psi_a.dag() * Ec * psi_a).real

print(f"  P(identify |ψ₁⟩) = {prob_a1:.4f}")
print(f"  P(identify |ψ₂⟩) = {prob_b1:.4f} (should be 0)")
print(f"  P(inconclusive) = {prob_c1:.4f}")
print(f"  Sum = {prob_a1 + prob_b1 + prob_c1:.4f}")

# Test on |ψ₂⟩
print(f"\n" + "=" * 50)
print("Testing on |ψ₂⟩")
print("=" * 50)

prob_a2 = (psi_b.dag() * Ea * psi_b).real
prob_b2 = (psi_b.dag() * Eb * psi_b).real
prob_c2 = (psi_b.dag() * Ec * psi_b).real

print(f"  P(identify |ψ₁⟩) = {prob_a2:.4f} (should be 0)")
print(f"  P(identify |ψ₂⟩) = {prob_b2:.4f}")
print(f"  P(inconclusive) = {prob_c2:.4f}")
print(f"  Sum = {prob_a2 + prob_b2 + prob_c2:.4f}")

print("\n" + "=" * 70)
print("KEY INSIGHTS")
print("=" * 70)
print("""
1. The USD POVM never makes an error!
   - When it identifies a state, it's always correct
   - The cost is a probability of inconclusive results

2. Optimal scaling α = 1/(1 + |⟨ψ₁|ψ₂⟩|)
   - Larger overlap → smaller success probability
   - Smaller overlap → larger success probability

3. Applications:
   - Quantum cryptography (eavesdropper detection)
   - Quantum communication
   - State discrimination tasks
""")

UNANIMOUS STATE DISCRIMINATION

States to discriminate:
  |ψ₁⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [0.]]
  |ψ₂⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.8660254]
 [0.5      ]]
  Overlap |⟨ψ₁|ψ₂⟩|² = 0.7500

Orthogonal states:
  |ψ₁_perp⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[0.]
 [1.]]
  |ψ₂_perp⟩ = Quantum object: dims=[[2], [1]], shape=(2, 1), type='ket', dtype=Dense
Qobj data =
[[ 0.5      ]
 [-0.8660254]]

Optimal scaling factor α = 0.5359

USD POVM elements:
  Ea (identifies |ψ₁⟩) =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.         0.        ]
 [0.         0.53589838]]

  Eb (identifies |ψ₂⟩) =
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.1339746  -0.23205081]
 [-0.23205081  0.40192379]]

  Ec (inconclusive) =
Quantum

## 6. Summary and Key Insights

### Projective vs POVM Measurements

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Feature</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Projective</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">POVM</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Operators</td>
            <td style="padding: 8px;">Projectors $P_m$</td>
            <td style="padding: 8px;">Positive operators $E_m$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Orthogonality</td>
            <td style="padding: 8px;">$P_m P_n = \delta_{mn} P_m$</td>
            <td style="padding: 8px;">Not required</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Completeness</td>
            <td style="padding: 8px;">$\sum_m P_m = I$</td>
            <td style="padding: 8px;">$\sum_m E_m = I$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Number of outcomes</td>
            <td style="padding: 8px;">≤ dimension</td>
            <td style="padding: 8px;">Can exceed dimension</td>
        </tr>
        <tr>
            <td style="padding: 8px;">State after measurement</td>
            <td style="padding: 8px;">Defined collapse</td>
            <td style="padding: 8px;">Not specified</td>
        </tr>
    </tbody>
</table>

### POVM Summary

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Concept</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Formula</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">POVM elements</td>
            <td style="padding: 8px;">$E_m \geq 0$, $\sum_m E_m = I$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Probability</td>
            <td style="padding: 8px;">$p(m) = \text{Tr}(E_m \rho)$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Naimark's theorem</td>
            <td style="padding: 8px;">$E_m = \langle 0|_A U^\dagger P_m U |0\rangle_A$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">USD optimal scaling</td>
            <td style="padding: 8px;">$\alpha = \frac{1}{1 + |\langle\psi_1|\psi_2\rangle|}$</td>
        </tr>
    </tbody>
</table>

### QuTiP Measurement Functions

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Function</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Description</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Returns</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">`measure_observable()`</td>
            <td style="padding: 8px;">Projective measurement</td>
            <td style="padding: 8px;">(eigenvalue, collapsed_state)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">`measure_povm()`</td>
            <td style="padding: 8px;">Single POVM measurement</td>
            <td style="padding: 8px;">(index, collapsed_state)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">`measurement_statistics_povm()`</td>
            <td style="padding: 8px;">All POVM outcomes</td>
            <td style="padding: 8px;">(outcomes, probabilities)</td>
        </tr>
    </tbody>
</table>

### Key Takeaways

1. **POVMs generalize projective measurements** by relaxing the orthogonality condition
2. **POVM elements are positive operators** that sum to identity
3. **Number of outcomes can exceed dimension** (e.g., 3 outcomes for a qubit)
4. **Naimark's theorem** shows any POVM can be realized as projective measurement + ancilla
5. **Unambiguous State Discrimination** is a key application where POVMs achieve zero-error identification
6. **QuTiP** provides tools for implementing POVM measurements via `measure_povm()`

---

**Day 8 Complete!** 🎉

You now understand:
- The limitations of projective measurements
- What POVMs are and why they are needed
- The mathematical definition of POVM elements
- Naimark's theorem (dilation theorem)
- How to implement POVMs in QuTiP
- Applications of POVMs in quantum information

Proceed to Day 9 when ready.